In [3]:
import os
from getpass import getpass
import backoff
from openai import OpenAI
from tqdm import tqdm
from supabase import create_client, Client
from dotenv import load_dotenv
from pathlib import Path
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

env_path = Path("../nextjs-app/.env.local")
load_dotenv(env_path)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
print("SUPABASE_URL: ",SUPABASE_URL)
print("SUPABASE_KEY: ",SUPABASE_KEY)
print("PINECONE_API_KEY: ",PINECONE_API_KEY)

c:\Projects\media-ai-app\vector-pipeline\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SUPABASE_URL:  https://lhaxjdvtpitgtjrjfioe.supabase.co
SUPABASE_KEY:  sb_publishable_SQbgB9gTwmlMoFTfjpXdlQ_D1QH5uxB
PINECONE_API_KEY:  pcsk_8VH1g_3qAy5BRTh69nbzRjo6zm5QL9wFdFo4hNJxB3j2yKqxhKwCrZmVWvagFAotYsePh


In [5]:
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def fetch_entire_table(supabase_client, table_name, select_columns="*", batch_size=1000):
    """
    Fetches all rows from a Supabase table in batches.
    
    Args:
        supabase_client: Initialized Supabase client
        table_name (str): Name of the table to fetch
        select_columns (str): Columns to select, default "*"
        batch_size (int): Number of rows to fetch per batch, default 1000
        
    Returns:
        List of rows (dicts)
    """
    all_rows = []
    offset = 0

    while True:
        batch = (
            supabase_client.table(table_name)
            .select(select_columns)
            .range(offset, offset + batch_size - 1)
            .execute()
            .data
        )
        if not batch:
            break
        all_rows.extend(batch)
        offset += batch_size

    return all_rows

In [11]:
# fetch movies
media_res  = fetch_entire_table(supabase, "media", "id,title,overview,popularity,rating,type,release_date")
genres_res  = fetch_entire_table(supabase, "genre", "genre_id,genre_name")
mapping_res  = fetch_entire_table(supabase, "genre_media_map", "media_id,genre_id")
print(f"Fetched {len(media_res)} movies")
print(f"Fetched {len(genres_res)} genres")
print(f"Fetched {len(mapping_res)} genre-media mappings")

Fetched 500 movies
Fetched 27 genres
Fetched 1119 genre-media mappings


In [12]:

pinecone = Pinecone(api_key=PINECONE_API_KEY)
index_name = "media-index"
if index_name not in pinecone.list_indexes().names():
        pinecone.create_index(
            name=index_name,
            dimension=768,
            spec=ServerlessSpec(
                cloud='aws',
                region='us-east-1'
            )
        )
index = pinecone.Index(index_name)
print(f"Pinecone index '{index_name}' is ready")

Pinecone index 'media-index' is ready


In [13]:
print(media_res[0])
print(genres_res[0])
#Create map obj for media_res and genres_res for easy lookup
media_map = {m["id"]: m for m in media_res}
genres_map = {g["genre_id"]: g["genre_name"] for g in genres_res}
movie_genres = {}
for gm in mapping_res:
    media_id = gm["media_id"]
    genre_name = genres_map.get(gm["genre_id"], "Unknown")
    if media_id in movie_genres:
        movie_genres[media_id].append(genre_name)
    else:
        movie_genres[media_id] = [genre_name]

{'id': 1290821, 'title': 'Shelter', 'overview': 'A man living in self-imposed exile on a remote island rescues a young girl from a violent storm, setting off a chain of events that forces him out of seclusion to protect her from enemies tied to his past.', 'popularity': 372.8761, 'rating': 6.787, 'type': 'movie', 'release_date': '1/28/2026'}
{'genre_id': 28, 'genre_name': 'Action'}


In [15]:
# --------------------------
# Prepare chunks for embedding
# --------------------------
chunks = []
for media_id, media in media_map.items():
    title = media.get("title", "")
    overview = media.get("overview", "")
    genre_list = movie_genres.get(media_id, [])
    text_to_embed = f"{title}. Genres: {', '.join(genre_list)}. Overview: {overview}. Type: {media.get('type', '')}"
    append_info = {"id": str(media_id), "text": text_to_embed, "metadata": {"media_id": media_id, "genres": genre_list, "title": title, "release_date": media.get("release_date", ""), "popularity": media.get("popularity", 0), "rating": media.get("rating", 0), "type": media.get("type", "")}}
    chunks.append(append_info)
print(chunks[0])

{'id': '1290821', 'text': 'Shelter. Genres: Action, Crime, Thriller. Overview: A man living in self-imposed exile on a remote island rescues a young girl from a violent storm, setting off a chain of events that forces him out of seclusion to protect her from enemies tied to his past.. Type: movie', 'metadata': {'media_id': 1290821, 'genres': ['Action', 'Crime', 'Thriller'], 'title': 'Shelter', 'release_date': '1/28/2026', 'popularity': 372.8761, 'rating': 6.787, 'type': 'movie'}}


In [16]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

BATCH_SIZE = 50
for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Upserting to Pinecone"):
    batch = chunks[i:i+BATCH_SIZE]
    texts = [c["text"] for c in batch]

    # Generate embeddings locally
    embeddings = model.encode(texts, show_progress_bar=False)

    # Upsert to Pinecone
    vectors = [
        (c["id"], embeddings[idx].tolist(), c["metadata"])
        for idx, c in enumerate(batch)
    ]
    index.upsert(vectors=vectors)

print("✅ All chunks upserted!")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8464.49it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Upserting to Pinecone: 100%|██████████| 10/10 [00:35<00:00,  3.58s/it]

✅ All chunks upserted!


In [ ]:
query_embedding = model.encode(["Find sci-fi movies like Interstellar"])[0].tolist()

results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
    
)

Query results: QueryResponse(matches=[{'id': '157336',
 'metadata': {'genres': ['Adventure', 'Drama', 'Science Fiction'],
              'media_id': 157336,
              'popularity': 46.6697,
              'rating': 8.468,
              'release_date': '11/5/2014',
              'title': 'Interstellar',
              'type': 'movie'},
 'score': 0.67944628,
 'values': []}, {'id': '434853',
 'metadata': {'genres': ['Science Fiction', 'Action', 'Thriller'],
              'media_id': 434853,
              'popularity': 36.7576,
              'rating': 5.7,
              'release_date': '5/1/2025',
              'title': 'Space/Time',
              'type': 'movie'},
 'score': 0.531583846,
 'values': []}, {'id': '945961',
 'metadata': {'genres': ['Horror', 'Science Fiction'],
              'media_id': 945961,
              'popularity': 21.606,
              'rating': 7.168,
              'release_date': '8/13/2024',
              'title': 'Alien: Romulus',
              'type': 'movie'},
 